In [1]:
import json
from crewai import Agent, Task, Crew, Process, LLM

# TUTORING FLOW

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph

In [2]:

# --- 1. Helper Function: File Reading ---
def read_text_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file {file_path}: {str(e)}"

# --- 2. Konfigurasi LLM ---
llm = LLM(
    model="ollama/llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.1
)

# --- 3. Load Knowledge Base ---
pseudocode_path = '/home/ilham/Documents/python/crewai-vs-langgraph/doc/Pseudocode dan Golang Dasar.md'
pseudocode_knowledge = read_text_file(pseudocode_path)

misconceptions_path = '/home/ilham/Documents/python/crewai-vs-langgraph/doc/List of misconceptions.md'
misconceptions_knowledge = read_text_file(misconceptions_path)

# --- 4. Definisi Sub-Agents ---

# Sub-Agent 1: Style & Syntax Checker
style_checker_agent = Agent(
    role="Style & Syntax Auditor",
    goal="Mendeteksi kesalahan format, deklarasi variabel, typo, dan kepatuhan standar penulisan pseudocode.",
    backstory=(
        "Anda adalah asisten dosen yang sangat teliti terhadap detail penulisan.\n"
        "Fokus Anda HANYA pada:\n"
        "1. Kamus variabel (deklarasi lengkap atau tidak)\n"
        "2. Tipe data yang sesuai\n"
        "3. Konsistensi nama variabel (typo antara kamus dan algoritma)\n"
        "4. Struktur dasar (program/endprogram, algoritma, dll)\n"
        "5. Format penulisan (indentasi, kapitalisasi)\n\n"
        "JANGAN analisis logika program - itu bukan tugas Anda.\n\n"
        f"Acuan Standar:\n{pseudocode_knowledge}"
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Sub-Agent 2: Logic & Algorithm Checker
logic_checker_agent = Agent(
    role="Algorithmic Logic Analyst",
    goal="Menganalisis logika algoritma dan membandingkan dengan solusi referensi untuk menemukan kesalahan alur.",
    backstory=(
        "Anda adalah ahli algoritma dan logika pemrograman.\n\n"
        "Tugas Anda:\n"
        "1. Bandingkan pseudocode siswa dengan context_solution\n"
        "2. Identifikasi kesalahan LOGIKA, bukan syntax/typo\n"
        "3. Fokus pada:\n"
        "   - Kondisi If/Else yang terbalik atau salah\n"
        "   - Logika perulangan yang tidak tepat\n"
        "   - Operasi matematika yang keliru\n"
        "   - Edge cases yang tidak tertangani\n"
        "   - Alur yang tidak efisien\n\n"
        "PENTING - Identifikasi Miskonsepsi:\n"
        "- HANYA identifikasi miskonsepsi jika benar-benar cocok dengan definisi\n"
        "- Kesalahan operator (=, ==) adalah SYNTAX ERROR, bukan miskonsepsi\n"
        "- Kondisi terbalik (if ganjil output genap) adalah LOGIC ERROR, bukan miskonsepsi\n"
        "- 'Intentional Bug' hanya untuk asumsi sistem bisa prediksi masa depan\n"
        "- 'While Demon' hanya untuk loop yang berhenti preventif\n\n"
        f"Referensi Miskonsepsi:\n{misconceptions_knowledge}\n\n"
        "ABAIKAN masalah typo/format - itu tugas Style Auditor."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# --- 5. Supervisor Agent ---
# Supervisor ini akan bekerja di akhir rangkaian (Sequential) untuk mengagregasi hasil.
scoring_supervisor_agent = Agent(
    role="Grading Supervisor",
    goal="Mengkonsolidasi laporan dari Style & Logic Auditor untuk menghasilkan skor final dan JSON output.",
    backstory=(
        "Anda adalah Kepala Penilai.\n\n"
        "Tugas Anda:\n"
        "1. MEMBACA laporan dari Style Auditor dan Logic Analyst yang diberikan dalam context.\n"
        "2. MENGHITUNG pengurangan poin berdasarkan rubrik.\n"
        "3. MENENTUKAN status 'correct' (benar/salah secara fungsional).\n"
        "4. MERANGKUM kesalahan dan miskonsepsi.\n"
        "5. MENGHASILKAN output JSON final.\n\n"
        "ATURAN PENILAIAN:\n"
        "- Mulai dari skor 100\n"
        "- Jangan double-count kesalahan yang sama\n"
        "- Prioritaskan kesalahan logika mayor\n"
        "- Syntax error minor: maksimal -10 poin total\n"
        "- Kondisi if/else terbalik: -40 poin (logika mayor)\n\n"
        "ATURAN OUTPUT:\n"
        "- Field 'pseudocode' HARUS berisi pseudocode ASLI siswa (tidak dimodifikasi)\n"
        "- Field 'Misconceptions' berisi array miskonsepsi atau [] jika tidak ada\n"
        "- Jangan tambahkan markdown ```json```\n"
        "- Output harus valid JSON"
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# --- 6. Definisi Tasks ---

# Task untuk Sub-Agent 1
style_analysis_task = Task(
    description=(
        "Analisis aspek STYLE dan SYNTAX dari pseudocode siswa.\n\n"
        "Pseudocode Siswa:\n{pseudocode}\n\n"
        "Cari dan laporkan:\n"
        "1. Deklarasi variabel yang hilang atau tidak lengkap\n"
        "2. Typo nama variabel antara kamus dan algoritma\n"
        "3. Tipe data yang tidak sesuai atau salah\n"
        "4. Format penulisan yang tidak standar\n"
        "5. Struktur program yang tidak lengkap\n\n"
        "JANGAN analisis logika - fokus hanya pada penulisan dan format.\n\n"
        "Output format:\n"
        "Kategori: [Style/Syntax]\n"
        "Kesalahan: [deskripsi singkat]\n"
        "Pengurangan: [perkiraan poin berdasarkan severity]"
    ),
    expected_output="Laporan terstruktur tentang kesalahan style dan syntax dengan estimasi pengurangan poin.",
    agent=style_checker_agent
)

# Task untuk Sub-Agent 2
logic_analysis_task = Task(
    description=(
        "Analisis LOGIKA ALGORITMA dari pseudocode siswa.\n\n"
        "Problem: {problem}\n\n"
        "Context Solution (Referensi):\n{context_solution}\n\n"
        "Pseudocode Siswa:\n{pseudocode}\n\n"
        "Langkah analisis:\n"
        "1. Bandingkan logika siswa dengan solusi referensi\n"
        "2. Identifikasi perbedaan kondisi, perulangan, atau operasi\n"
        "3. Tentukan apakah perbedaan tersebut menghasilkan output yang SALAH\n"
        "4. Kategorikan kesalahan: Kritis/Mayor/Minor\n"
        "5. Periksa miskonsepsi HANYA jika benar-benar sesuai definisi\n\n"
        "Untuk problem Ganjil/Genap:\n"
        "- Cek apakah kondisi mod 2 benar\n"
        "- Cek apakah output sesuai dengan kondisi\n"
        "- Kondisi terbalik = Logic Error Mayor, BUKAN miskonsepsi\n\n"
        "Output format:\n"
        "Kategori: [Logika Kritis/Mayor/Minor]\n"
        "Kesalahan: [deskripsi dengan perbandingan ke solusi]\n"
        "Miskonsepsi: [nama miskonsepsi dari list atau 'Tidak Ada']\n"
        "Pengurangan: [perkiraan poin]"
    ),
    expected_output="Laporan terstruktur tentang kesalahan logika, perbandingan dengan solusi, dan identifikasi miskonsepsi yang akurat.",
    agent=logic_checker_agent
)

# Task untuk Supervisor (Koordinasi & Final Scoring)
final_grading_task = Task(
    description=(
        "Sebagai Supervisor, koordinasikan penilaian dan hasilkan skor final.\n\n"
        "Input yang tersedia:\n"
        "- Problem: {problem}\n"
        "- Context Solution: {context_solution}\n"
        "- Pseudocode Siswa: {pseudocode}\n"
        "- Rubrik: {general_rubrication}\n\n"
        "Langkah kerja:\n"
        "1. BACA dan PELAJARI laporan dari Style Auditor dan Logic Analyst (tersedia di context).\n"
        "2. JANGAN mencoba mengerjakan ulang analisis mereka, percayai laporan mereka.\n"
        "3. HITUNG total pengurangan poin berdasarkan laporan tersebut:\n"
        "   - Hindari double counting\n"
        "   - Prioritaskan kesalahan terberat\n"
        "4. TENTUKAN 'correct': false jika output salah, true jika benar\n"
        "5. RANGKUM dalam JSON\n\n"
        "CRITICAL - Format JSON WAJIB:\n"
        "```\n"
        "{\n"
        '  "score": <angka integer saja>,\n'
        '  "correct": <true atau false>,\n'
        '  "summary": "<penjelasan singkat hasil penilaian>",\n'
        '  "Misconceptions": [<array string miskonsepsi atau [] jika kosong>],\n'
        '  "pseudocode": "<COPY EXACT pseudocode asli siswa TANPA MODIFIKASI>"\n'
        "}\n"
        "```\n\n"
        "VALIDASI AKHIR:\n"
        "- Pastikan field 'pseudocode' = pseudocode siswa asli (cek dengan input)\n"
        "- Jangan tampilkan pseudocode yang sudah diperbaiki\n"
        "- Jangan tambahkan ```json``` wrapper\n"
        "- Output harus bisa di-parse sebagai JSON. HANYA JSON. JANGAN ada teks lain."
    ),
    expected_output="Valid JSON string dengan format yang sudah ditentukan.",
    agent=scoring_supervisor_agent,
    context=[style_analysis_task, logic_analysis_task]  # Supervisor menerima hasil kedua task
)

# --- 7. Crew Process ---
# OPTIMIZATION: Menggunakan Process.sequential agar "tidak terlalu berat" dan lebih stabil.
# Meskipun sequential, struktur peran tetap "Hierarchical" karena ada Supervisor yang mengagregasi 
# output dari sub-agents (Style & Logic) di akhir proses.
grading_crew = Crew(
    agents=[
        style_checker_agent,
        logic_checker_agent,
        scoring_supervisor_agent
    ],
    tasks=[
        style_analysis_task,
        logic_analysis_task,
        final_grading_task
    ],
    process=Process.sequential, # Changed to Sequential for performance and stability with local LLM
    verbose=True
)

# --- 8. Data Input & Eksekusi ---
input_data = {
    'problem': "Buatlah algoritma untuk menentukan apakah sebuah bilangan N adalah Ganjil atau Genap. Output 'Ganjil' atau 'Genap'.",
    
    'context_solution': """
        Program GanjilGenap
        kamus
            N : integer
        algoritma
            input(N)
            if (N mod 2 == 0) then
                output("Genap")
            else
                output("Ganjil")
            endif
        endprogram
    """,
    
    'pseudocode': """
        Program CekBilangan
        kamus
            bil : integer
        algoritma
            input(bil)
            if (bil mod 2 = 1) then
                print("Genap")  
            else
                print("Ganjil")
        endprogram
    """,
    
    'general_rubrication': """
        Start Score: 100.
        Pengurangan:
        1. Kesalahan Kritis:
        - Logika inti sepenuhnya salah, menghasilkan output tidak relevan: -50 Poin
        - Program tidak menyelesaikan masalah sama sekali (ada usaha): -90 Poin
        - Lembar jawaban kosong: -50 Poin

        2. Kesalahan Logika Mayor:
        - Gagal menangani salah satu kondisi utama: -40 Poin
        - Perhitungan matematis utama tidak akurat: -40 Poin
        - Variabel tidak tertulis pada kamus: -25 Poin

        3. Kesalahan Logika Minor & Struktur:
        - Gagal menangani kasus khusus (edge case): -25 Poin
        - Tidak menggunakan tipe bentukan (jika diwajibkan): -25 Poin
        - Alur program tidak efisien/berbelit: -15 Poin

        4. Kesalahan Kelengkapan:
        - Tipe data variabel tidak sesuai: -10 Poin
        - Penulisan variabel berbeda (typo) antara kamus & program: -5 Poin
        - Format output tidak sesuai: -5 Poin
    """
}

print("="*80)
print("### MEMULAI PROSES PENILAIAN (OPTIMIZED SEQUENTIAL) ###")
print("="*80)

result = grading_crew.kickoff(inputs=input_data)

print("\n" + "="*80)
print("## HASIL JSON FINAL ##")
print("="*80 + "\n")

# Clean up output
clean_result = str(result).replace("```json", "").replace("```", "").strip()

# Validasi dan output
try:
    json_output = json.loads(clean_result)
    
    # Validasi tambahan
    print("✓ JSON berhasil di-parse\n")
    
    # Cek pseudocode
    original_pseudo = input_data["pseudocode"].strip()
    output_pseudo = json_output.get("pseudocode", "").strip()
    
    if original_pseudo != output_pseudo:
        print("! WARNING: Pseudocode di output berbeda dengan input asli!")
        print(f"   Panjang original: {len(original_pseudo)} chars")
        print(f"   Panjang output: {len(output_pseudo)} chars\n")
    else:
        print("✓ Pseudocode di output sesuai dengan input asli\n")
    
    # Cek miskonsepsi
    misconceptions = json_output.get("Misconceptions", [])
    if not misconceptions or misconceptions == []:
        print("✓ Tidak ada miskonsepsi teridentifikasi (sesuai ekspektasi)\n")
    else:
        print(f"! Miskonsepsi terdeteksi: {misconceptions}\n")
    
    # Cek skor
    score = int(json_output.get("score", 0))
    expected_range = (55, 65)  # Range wajar untuk kasus ini
    if expected_range[0] <= score <= expected_range[1]:
        print(f"✓ Skor {score} dalam range wajar {expected_range}\n")
    else:
        print(f"! Skor {score} di luar range wajar {expected_range}\n")
    
    # Output JSON
    print("-" * 80)
    print(json.dumps(json_output, indent=2, ensure_ascii=False))
    print("-" * 80)
    
except json.JSONDecodeError as e:
    print("X ERROR: Gagal parsing JSON")
    print(f"   Error: {e}\n")
    print("Raw Output:")
    print("-" * 80)
    print(clean_result)
    print("-" * 80)
except Exception as e:
    print(f"X ERROR: {e}\n")
    print("Raw Output:")
    print("-" * 80)
    print(clean_result)
    print("-" * 80)


### MEMULAI PROSES PENILAIAN (OPTIMIZED SEQUENTIAL) ###


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a1d1fd04-f56a-4088-9d0d-c4be26a30093                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Task: Analisis aspek STYLE dan SYNTAX dari pseudocode siswa.                                                   │
│                                                                                                                 │
│  Pseudocode Siswa:                                                                                              │
│                                                                                                                 │
│          Program CekBilangan                                                                                    │
│          kamus                                                                                                  │
│              bil : integer                                                                                      │
│          algoritma                                                                                              │
│              input(bil)                                                                                         │
│              if (bil mod 2 = 1) then                                                                            │
│                  print("Genap")                                                                                 │
│              else                                                                                               │
│                  print("Ganjil")                                                                                │
│          endprogram                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│  Cari dan laporkan:                                                                                             │
│  1. Deklarasi variabel yang hilang atau tidak lengkap                                                           │
│  2. Typo nama variabel antara kamus dan algoritma                                                               │
│  3. Tipe data yang tidak sesuai atau salah                                                                      │
│  4. Format penulisan yang tidak standar                                                                         │
│  5. Struktur program yang tidak lengkap                                                                         │
│                                                                                                                 │
│  JANGAN analisis logika - fokus hanya pada penulisan dan format.                                                │
│                                                                                                                 │
│  Output format:                                                                                                 │
│  Kategori: [Style/Syntax]                                                                                       │
│  Kesalahan: [deskripsi singkat]                                                                                 │
│  Pengurangan: [perkiraan poin berdasarkan severity]                                                             │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

/home/ilham/Documents/python/crewai-vs-langgraph/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Kategori: Style                                                                                                │
│  Kesalahan: Deklarasi variabel "bil" tidak lengkap dan typo nama variabel antara kamus dan algoritma.           │
│  Pengurangan: 5 poin (2 poin untuk deklarasi variabel yang hilang, 1 poin untuk typo nama variabel, 1 poin      │
│  untuk tipe data yang tidak sesuai, 1 poin untuk format penulisan yang tidak standar)                           │
│                                                                                                                 │
│  Kategori: Syntax                                                                                               │
│  Kesalahan: Struktur program yang tidak lengkap dan kurangnya deklarasi variabel.                               │
│  Pengurangan: 3 poin (2 poin untuk struktur program yang tidak lengkap, 1 poin untuk kurangnya deklarasi        │
│  variabel)                                                                                                      │
│                                                                                                                 │
│  Total pengurangan: 8 poin                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


## HASIL JSON FINAL ##

X ERROR: Gagal parsing JSON
   Error: Extra data: line 9 column 1 (char 342)

Raw Output:
--------------------------------------------------------------------------------
{
  "score": 84,
  "correct": false,
  "summary": "Kesalahan logika mayor dan minor ditemukan dalam pseudocode siswa.",
  "Misconceptions": [],
  "pseudocode": "Program CekBilangan\nkamus\n    bil : integer\nalgoritma\n    input(bil)\n    if (bil mod 2 = 1) then\n        print(\"Genap\")\n    else\n        print(\"Ganjil\")\nendprogram"
}

Catatan: Pseudocode di atas adalah pseudocode asli siswa, tidak dimodifikasi.
--------------------------------------------------------------------------------


/home/ilham/Documents/python/crewai-vs-langgraph/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 11e2ed3e-96ef-4c44-9cfd-8b1b0268a92a                                                                     │
│  Agent: Grading Supervisor                                                                                      │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a1d1fd04-f56a-4088-9d0d-c4be26a30093                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```                                                                                              │
│  {                                                                                                              │
│    "score": 84,                                                                                                 │
│    "correct": false,                                                                                            │
│    "summary": "Kesalahan logika mayor dan minor ditemukan dalam pseudocode siswa.",                             │
│    "Misconceptions": [],                                                                                        │
│    "pseudocode": "Program CekBilangan\nkamus\n    bil : integer\nalgoritma\n    input(bil)\n    if (bil mod 2   │
│  = 1) then\n        print(\"Genap\")\n    else\n        print(\"Ganjil\")\nendprogram"                          │
│  }                                                                                                              │
│  ```                                                                                                            │
│  Catatan: Pseudocode di atas adalah pseudocode asli siswa, tidak dimodifikasi.                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph